In [ ]:
import pprint

In [ ]:
from litellm import completion

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather in a given location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "The unit of temperature.",
                    },
                },
                "required": ["location"],
            }
        }
    }
]

messages = [{"role": "user", "content": "What's the weather like in Boston today?"}]

response = completion(
    model="openai/unsloth/gemma-4-E4B-it-GGUF:Q8_0",  # name can be anything locally
    api_base="http://localhost:8080/v1",
    api_key="not-needed",   # llama.cpp doesn't require it
    messages=messages,
    tools=tools,
)

print(response.choices[0].message.tool_calls[0].function)


In [ ]:
!pip install llama_index
!pip install llama_index.llms.litellm

In [ ]:
import nest_asyncio
from llama_index.llms.litellm import LiteLLM
from llama_index.core.agent.workflow import ReActAgent, AgentStream
from llama_index.core.tools import FunctionTool

nest_asyncio.apply()

def add(a:int|float, b:int|float) -> int|float:
    """Adds 2 integers or floats"""
    return a+b

llm = LiteLLM(
    model="openai/unsloth/gemma-4-E4B-it-GGUF:Q8_0",
    api_base="http://localhost:8080/v1",
    api_key="sk-no-key-required"
)

agent = ReActAgent(
    tools=[FunctionTool.from_defaults(add)],
    llm=llm
)
handler = agent.run("What is 21.921 + 23.21?")
async for event in handler.stream_events():
    if isinstance(event, AgentStream):
        print(event.delta, end="", flush=True)

In [ ]:
import httpx, os

key = os.getenv("Z_AI_API_KEY")

r = httpx.post(
    "https://api.z.ai/api/anthropic/v1/messages",
    headers={
        "x-api-key": key,
        "content-type": "application/json"
    },
    json={
        "model": "glm-5.1",
        "messages": [{"role": "user", "content": "test"}],
        "max_tokens": 10
    },
    timeout=30
)

print(r.status_code)
print(r.text)

In [ ]:
from litellm import completion

response = completion(
    model="zai/glm-4.7",
    api_key="ac6c62cbda4c4326a108fc38f6461b3f.iTVMYwyDJx2ODRod",   # 🔥 force it here
    api_base="https://api.z.ai/api/coding/paas/v4",
    messages=[
        {"role": "user", "content": "Hello"}
    ]
)

print(response)

In [ ]:
key = os.getenv("ZAI_API_KEY")
key

In [ ]:
import httpx, os

key = os.getenv("ZAI_API_KEY")

r = httpx.post(
    "https://api.z.ai/api/coding/paas/v4/chat/completions",
    headers={
        "Authorization": f"Bearer {key}",
        "Content-Type": "application/json"
    },
    json={
        "model": "glm-5.1",
        "messages": [
            {"role": "user", "content": "what is 1 + 1"}
        ],
        "max_tokens": 100
    },
    timeout=30
)

data = r.json()
msg = data["choices"][0]["message"]

print("Answer:", msg.get("content"))
print("Reasoning:", msg.get("reasoning_content"))